In [ ]:
from os import listdir as ls
import pandas as pd
import json
import pickle
import pycountry
import pycountry_convert as pc
import arviz as az
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown, Latex

from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, WANING_COMPARISON, WANING_TIMEOUTS, WANING_RERUNS, \
    MOB_LOCATION_SOURCE_MAP, MOB_LOCATION_NAME_MAP, CONT_CMAP
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import load_targets, get_country_procs
from emu_renewal.plotting import plot_multianalysis_fit, add_cont_to_world_geodf, plot_continent_grouping, plot_prior_multipost, \
    compare_proc_mob, compare_proc_weighted_gmob, plot_proc_comparison
from emu_renewal.utils import get_countries_by_continent, get_analysis_paths, get_analysis_commits_df, split_list_into_segments

set_matplotlib_formats("svg")

# Purpose
This document presents the calibration to data
for each country, analysis and target indicator for fitting.

In [ ]:
display(Latex(r"\newpage"))

# Country grouping
For this document and the following supplemental files,
we grouped countries according to the following continent groupings.

In [ ]:
#| fig-align: center
world = get_world_shp()
add_cont_to_world_geodf(world)
plt.style.use("default")
plot_continent_grouping(world)

In [ ]:
display(Latex(r"\newpage"))

# Results by continent and country

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
# analysis_paths = get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
countries_by_cont = {"AF": ["DZA", "AGO"]} # *** TEMPORARY PATCH TO REDUCE RUN TIME FOR NOTEBOOK ***
for cont, countries in countries_by_cont.items():
    display(Latex(r"\newpage"))
    display(Markdown(f"## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = analysis_paths[country]
        
        no_mob_path = analyses["no_mob"]
    
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analyses.items()}
        targs = load_targets(no_mob_path)
        display(plot_multianalysis_fit(targs, spaghs))

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())

# Purpose
This document presents comparisons of parameter prior distributions 
to their corresponding posterior distributions.

In [ ]:
display(Latex(r"\newpage"))

# Prior posterior comparisons by continent and country

In [ ]:
#| fig-align: center
for co, (cont, countries) in enumerate(countries_by_cont.items()):
    if co:
        display(Latex(r"\newpage"))
    display(Markdown(f"\n## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = analysis_paths[country]

        no_mob_path = analyses["no_mob"]
        targs = load_targets(no_mob_path)
            
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
        var_names = ["start"] + [v[5:] for v in targs.keys() if v.startswith("prop_")]
        display(plot_prior_multipost(var_names, priors, idatas, 4))

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())

In [ ]:
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

# Purpose
This document presents the residual transmission scaling (with uncertainty) implemented without scaling for mobility for each country. It is intended to provide a raw comparison between the residual non-mechanistic transmission that needed to be applied for each analysis in the absence of mobility scaling and the various mobility data fields available from both Google and Facebook. We also show comparisons against the composite Google mobility metric after weighting, with associated credible intervals.

In [ ]:
display(Markdown("\\newpage # Individual mobility metric comparisons (Google and Facebook)\n\n"))
for m, mob_type in enumerate(MOB_LOCATION_SOURCE_MAP):
    mob_source = MOB_LOCATION_SOURCE_MAP[mob_type]
    mob_name = MOB_LOCATION_NAME_MAP[mob_type]
    if m:
        display(Latex(r"\newpage"))
    section_title = f"## {mob_name} mobility metric comparison\n\n"
    display(Markdown(section_title))
    display(Markdown(notes[mob_source]))
    mob_avail_countries = [c for c in all_countries if mob_source in analysis_paths[c]]
    for cont in ["AF"]:
    # for cont in CONT_CMAP:
        cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
        display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
        for countries in split_list_into_segments(cont_countries, 12):
            display(compare_proc_mob(analysis_paths, countries, 4, mob_type))

mob_avail_countries = [c for c in all_countries if "g_mob" in analysis_paths[c]]
display(Markdown("# Weighted scaling process versus residual comparison\n\n"))
for cont in ["AF"]:
# for cont in CONT_CMAP:
    display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
    
    cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
    for countries in split_list_into_segments(cont_countries, 16):
        display(compare_proc_weighted_gmob(analysis_paths, countries, 200, 4))

# Purpose
This document presents comparisons of residual transmission scaling
across each mobility analysis approach for each country,
along with the 95% credible interval associated with each.
We would expect that approaches to analysis that
if applying mobility captured some part of the variation 
in transmission that would otherwise need to be 
included through residual variation,
then that mobility approach would be associated with
less dramatic changes in the residual transmission scaling.

# Results by continent

In [ ]:
procs = get_country_procs(analysis_paths, all_countries)
for cont in countries_by_cont:
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    cont_countries = countries_by_cont[cont]
    for countries in split_list_into_segments(cont_countries, 9):
        display(plot_proc_comparison(procs, countries, analysis_paths))

{{< pagebreak >}}

# Commits used for analyses
For reproducbility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())